In [3]:
import nltk

def ngrams(sentence, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)

sentence = '안녕하세요. 만나서 진심으로 반가워요.'

unigram = ngrams(sentence, 1)
bigram = ngrams(sentence, 2)
trigram = ngrams(sentence, 3)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))


[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]


# 벡터화

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    'That movie is famous movie',
    'I like that actor',
    "I don't like that actor"
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [ ]:
# 맥용
# import os
# os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

In [5]:
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_dim
        )
        self.linear = nn.Linear(
            in_features = embedding_dim,
            out_features = vocab_size
        )
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [6]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 56.0MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 33.3MB/s]                            


In [7]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [8]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus = tokens, n_vocab = 5000, special_tokens = ['<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [9]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words  = sentence[window_start:idx] + sentence[idx + 1 : window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs

word_pairs = get_word_pairs(tokens, window_size = 2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [ ]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id['<unk>']
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))   

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:,0]
context_indexes = index_pairs[:,1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [18]:
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

word2vec = VanillaSkipgram(vocab_size = len(token_to_id), embedding_dim = 128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)

cuda


In [19]:
import torch.optim as optim

for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss
    cost = cost / len(dataloader)
    print(epoch + 1, cost)

1 tensor(6.1955, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(5.9813, device='cuda:0', grad_fn=<DivBackward0>)
3 tensor(5.9318, device='cuda:0', grad_fn=<DivBackward0>)
4 tensor(5.9016, device='cuda:0', grad_fn=<DivBackward0>)
5 tensor(5.8796, device='cuda:0', grad_fn=<DivBackward0>)
6 tensor(5.8617, device='cuda:0', grad_fn=<DivBackward0>)
7 tensor(5.8472, device='cuda:0', grad_fn=<DivBackward0>)
8 tensor(5.8341, device='cuda:0', grad_fn=<DivBackward0>)
9 tensor(5.8227, device='cuda:0', grad_fn=<DivBackward0>)
10 tensor(5.8122, device='cuda:0', grad_fn=<DivBackward0>)


In [20]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding
index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)


연기
tensor([-0.9941,  0.0095,  0.0589,  0.1554, -0.1830,  0.1026, -0.4612,  0.3336,
         1.7943,  0.0266, -2.2352, -1.7360,  1.0058, -0.1402,  1.2386,  0.6967,
        -0.6222,  0.1576,  0.9649,  0.7528, -0.4915, -0.4065, -0.5284,  1.2206,
        -0.2786,  0.5071,  0.6861,  0.8035,  0.7851,  1.2102,  1.7735,  0.1594,
         0.9961,  1.2637, -0.6497, -1.4338,  0.1182, -1.2316, -0.4225, -0.7056,
        -0.7469,  0.9510,  0.1631, -0.2984, -1.1968,  0.9631,  0.4010,  1.1554,
        -1.2045, -1.3581,  1.0688,  1.4749,  0.4299, -1.0110, -0.2202,  1.3776,
         0.1184,  0.9016, -0.9320, -1.0388, -0.4192, -1.1208, -0.2331,  1.5564,
         0.5561, -1.6713,  0.3126, -0.1352,  1.0113,  0.5396,  0.6276,  0.6802,
        -0.8147, -0.9416,  1.0303, -0.3327,  0.6646, -0.7323, -0.1899,  0.2425,
        -1.0612,  1.2933, -0.2672,  1.3138,  0.1233, -1.5729, -0.1063,  1.2022,
         0.7059,  0.5985,  0.3713,  1.9596,  1.6593, -0.0940, -0.5376,  0.9491,
         0.4123,  0.7187, -0.8053, -0

In [21]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis =1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1: n + 1]
    return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n = 5)

for index in top_n:
    print(id_to_token[index], cosine_matrix[index])

연기력 0.33664376
한테 0.29981428
황금 0.27102485
트랜스포머 0.2693696
전도연 0.26578182


# 26. 9. 16

In [4]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')

# DataFrame으로 만들기
df = pd.DataFrame({
    'text': corpus.test.texts,
    'label': corpus.test.labels
})
print(df.head())


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\wm032\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\wm0

In [7]:
tokenizer = Okt()                                              # tokennizer → tokenizer
tokens = [tokenizer.morphs(review) for review in corpus.train.texts]  # corpus.text → corpus.train.texts

In [10]:
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences = tokens,
    vector_size= 128,
    window = 5,
    min_count = 1,
    sg = 1,
    epochs = 3,
    max_final_vocab = 10000
)

In [11]:
word2vec.save('./models/word2vec.model')
word2vec = Word2Vec.load('./models/word2vec.model')

In [12]:
word = '연기'
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn = 5))
print(word2vec.wv.similarity(w1 = word, w2 = '연기력'))

[-0.30657566 -0.34346202  0.23212206 -0.4462044   0.35615173 -0.18104088
 -0.12027171  0.09781867 -0.04727696 -0.19009699  0.18384942 -0.01205717
  0.17833334  0.2962219  -0.19209117  0.13236998  0.18957806  0.47343883
  0.30663902  0.29713404 -0.18748632  0.43307334  0.5370091  -0.2512686
 -0.03192876  0.5374548  -0.10805429 -0.03374063  0.17523742 -0.09948366
 -0.30091384  0.28900412  0.16281773  0.06057506  0.08169246  0.36579332
  0.13894475 -0.39094222 -0.1267705  -0.25942388 -0.00286479 -0.09589416
  0.03642457 -0.13242236  0.12241141 -0.31209594 -0.27767485 -0.2691888
  0.426759    0.02793176  0.33138198  0.8412152   0.1333933  -0.1760284
 -0.35070235 -0.03535108  0.46049374  0.4039027   0.40688303  0.29043245
  0.20108786  0.08565689 -0.20632775 -0.28750348  0.2759797   0.15686114
  0.1742902   0.36079153 -0.16286664 -0.41129205  0.30499947 -0.42455667
 -0.51861745 -0.21774568  0.39459828 -0.5184317  -0.67581546  0.0919441
 -0.39463806  0.3486655  -0.14010079 -0.7122229   0.905

In [13]:
from Korpora import Korpora

corpus = Korpora.load('kornli')
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

[kornli] download multinli.train.ko.tsv: 83.6MB [00:05, 14.9MB/s]                            
[kornli] download snli_1.0_train.ko.tsv: 78.5MB [00:01, 73.1MB/s]                            
[kornli] download xnli.dev.ko.tsv: 516kB [00:00, 6.72MB/s]
[kornli] download xnli.test.ko.tsv: 1.04MB [00:00, 9.87MB/s]                           


[['개념적으로', '크림', '스키밍은', '제품과', '지리라는', '두', '가지', '기본', '차원을', '가지고', '있다.'], ['시즌', '중에', '알고', '있는', '거', '알아?', '네', '레벨에서', '다음', '레벨로', '잃어버리는', '거야', '브레이브스가', '모팀을', '떠올리기로', '결정하면', '브레이브스가', '트리플', 'A에서', '한', '남자를', '떠올리기로', '결정하면', '더블', 'A가', '그를', '대신하러', '올라가고', 'A', '한', '명이', '그를', '대신하러', '올라간다.'], ['우리', '번호', '중', '하나가', '당신의', '지시를', '세밀하게', '수행할', '것이다.']]


In [15]:
from gensim.models import FastText

fastText = FastText(
    sentences = tokens,
    vector_size = 128,
    window = 5,
    min_count = 5,
    sg = 1,
    max_final_vocab = 20000,
    epochs = 3,
    min_n = 2,
    max_n = 6
)

In [16]:
oov_token = '사랑해요'
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn = 5))

False
[('사랑', 0.8795445561408997), ('사랑에', 0.8236542344093323), ('사랑의', 0.7958014607429504), ('사랑을', 0.7568530440330505), ('사랑하는', 0.7218511700630188)]


# 양방향 다층신경망

In [12]:
# 양방향 다층신경망을 활용한
# 문장을 가지고 RNN을 해보는것

import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
input_size = 128
output_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(input_size = input_size,
               hidden_size = output_size,
               num_layers = num_layers,
               nonlinearity = "tanh",
               batch_first = True,
               bidirectional = bidirectional).to(device)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size).to(device)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 output_size).to(device)
outputs, hidden = model(inputs, h_0)

print(outputs.shape)
print(hidden.shape)
print(outputs.device)

torch.Size([4, 6, 512])
torch.Size([6, 4, 256])
cuda:0


In [3]:
input_size = 128
output_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size = input_size,
    hidden_size = output_size,
    num_layers = num_layers,
    batch_first = True,
    bidirectional = bidirectional,
    proj_size = proj_size
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional)+ 1),
    batch_size,
    proj_size if proj_size > 0 else output_size
)

c_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, output_size)
outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


c:\Users\wm032\Documents\computervision\.venv\Lib\site-packages\torch\nn\modules\rnn.py:1124: UserWarning: LSTM with projections is not supported with oneDNN. Using default implementation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\RNN.cpp:1474.)
  result = _VF.lstm(


In [10]:
import torch.nn as nn

class SentenceClassifier(nn.Module):
    def __init__(
            self,
            n_vocab,
            hidden_dim,
            embedding_dim,
            n_layers,
            dropout = 0.5,
            bidirectional = True,
            model_type = 'lstm'
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings = n_vocab,
            embedding_dim = embedding_dim,
            padding_idx = 0
        )
        if model_type == 'rnn':
            self.model = nn.RNN(
                input_size = embedding_dim,
                hidden_size = hidden_dim,
                num_layers = n_layers,
                bidirectional = bidirectional,
                dropout = dropout,
                batch_first = True
            )
        elif model_type == 'lstm':
            self.model = nn.LSTM(
                input_size = embedding_dim,
                hidden_size = hidden_dim,
                num_layers = n_layers,
                bidirectional = bidirectional,
                dropout = dropout,
                batch_first = True
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

In [1]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
corpus_df = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\wm032\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\wm0

In [4]:
# 노트북 셀에서 직접
!pip install tabulate


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\wm032\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [3]:
train = corpus_df.sample(frac = 0.9, random_state = 42)
test = corpus_df.drop(train.index)

print(train.head())
print(len(train))
print(len(test))

                                                    text  label
33553  모든 편견을 날려 버리는 가슴 따뜻한 영화. 로버트 드 니로, 필립 세이모어 호프만...      1
9427                    무한 리메이크의 소재. 감독의 역량은 항상 그 자리에...      0
199                                          신날 것 없는 애니.      0
12447                                              잔잔 격동      1
39489                                 오랜만에 찾은 주말의 명화의 보석      1
45000
5000


In [4]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]   # reivew → review

vocab = build_vocab(corpus = train_tokens, n_vocab = 5000, special_tokens = ['<pad>', '<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

In [6]:
import numpy as np

def pad_sequence(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)
    return np.asarray(result)

unk_id = token_to_id['<unk>']

# 각 리뷰를 번호 시퀀스로
train_ids = [
    [token_to_id.get(token, unk_id) for token in review]   # 안쪽: 토큰→번호
    for review in train_tokens                              # 바깥: 각 리뷰
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]

max_length = 32
pad_id = token_to_id['<pad>']
train_ids = pad_sequence(train_ids, max_length, pad_id)   # pad_sequence, train_ids
test_ids = pad_sequence(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [7]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype = torch.float32)
test_labels = torch.tensor(test.label.values, dtype = torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

In [8]:
print(len(train_loader))
print(len(test_loader))

2813
313


In [13]:
import torch.optim as optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

classifier = SentenceClassifier(n_vocab = n_vocab,
                                hidden_dim = hidden_dim,
                                embedding_dim = embedding_dim,
                                n_layers = n_layers).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr = 0.0001)

In [14]:
def train(model, datasets, criterion, optimizer, device, interval):   # 인자 + 콜론
    model.train()                    # 학습 모드
    losses = list()                  # 들여쓰기 (함수 안)

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if step % interval == 0:
        print(step, np.mean(losses))

def test(model,datasets, criterion, device):
    model.eval()
    losses = list()
    corrects = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits) > .5
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )

    print(np.mean(losses), np.mean(corrects))

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)

0.6924049022098699 0.5242
0.5765705112451182 0.7042
0.49444784443028056 0.7622
0.4760024533294641 0.7808
0.43655557920948 0.7992


In [15]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
    token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

보고싶다 [ 2.75712311e-01  6.87164426e-01  5.40673733e-01  1.04092948e-01
  1.41688490e+00 -8.91015470e-01 -1.64669347e+00  2.17219383e-01
 -9.95726466e-01 -1.05407560e+00 -1.69579685e+00 -1.30655682e+00
  1.53262913e-01 -2.37832761e+00  6.80614293e-01  5.52186072e-01
  4.97578472e-01 -5.00784576e-01 -2.24538112e+00 -1.61949480e+00
  1.20731354e+00 -1.25130862e-01  3.58285993e-01 -1.01710594e+00
  1.60015655e+00  3.88261914e-01 -4.42564011e-01 -1.61836028e+00
  2.72914559e-01 -2.23610473e+00 -2.11244655e+00  8.28484353e-03
  4.76476312e-01  1.04408407e+00 -6.23800993e-01 -1.54093719e+00
  4.32036817e-01 -7.47257650e-01 -9.86721992e-01  8.74805570e-01
  6.03228927e-01 -1.55447721e+00 -4.35993463e-01 -2.43830705e+00
 -1.28418243e+00  1.93590760e+00 -1.62210062e-01 -5.13640940e-01
  5.83372653e-01 -4.45307009e-02  1.46774733e+00 -1.62828267e+00
  6.56693161e-01 -3.29148680e-01  8.17698985e-03 -4.40081060e-01
  1.21621031e-03 -8.39917362e-01  1.16961151e-01  1.41581023e+00
  1.21738710e-01  1.